In [20]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import os, subprocess
if os.path.isdir('/content/NETRA/.git'):
    print('repo already present')
else:
    r = subprocess.run(['git', 'clone',
                        'https://github.com/varun250727108-oss/NETRA.git',
                        '/content/NETRA'], capture_output=True, text=True)
    print('clone exit code:', r.returncode)
    if r.returncode != 0:
        print(r.stderr)
        print('>>> If this failed, the repo is private: use the key-icon secret + '
              'token clone from earlier, then re-run this cell.')

repo already present


In [22]:
import hashlib, pathlib, shutil

SRC = pathlib.Path('/content/drive/MyDrive/netra_models')
ROOT = pathlib.Path('/content/NETRA')
EXPECT = {
    'netra_roi.pt':     '271aafe0e34dca31f3aa2cfa75d717e2c8fc808b00da22a14c6f61e79ee9eab5',
    'netra_roi.tflite': 'a581ff30c840b3ea4962b7bc584150e9b0499d369af42753102af958da6dec42',
}
targets = {
    'netra_roi.pt':     ROOT / 'core/models/netra_roi.pt',
    'netra_roi.tflite': ROOT / 'apps/mobile/android/app/assets/yolo26n_roi.tflite',
}
for name, dst in targets.items():
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(SRC / name, dst)
    h = hashlib.sha256(dst.read_bytes()).hexdigest()
    print(f'{name}: {dst.stat().st_size:,} bytes  sha256',
          'OK' if h == EXPECT[name] else f'MISMATCH -> {h}')

netra_roi.pt: 5,392,197 bytes  sha256 OK
netra_roi.tflite: 9,837,597 bytes  sha256 OK


In [24]:
import os
print('repo cloned:', os.path.isdir('/content/NETRA/.git'))
os.makedirs('/content/NETRA/tools', exist_ok=True)
os.makedirs('/content/NETRA/core/fixtures/yolo', exist_ok=True)
print('directories ready')

repo cloned: True
directories ready


In [25]:
%%writefile /content/NETRA/tools/record_yolo_golden.py
#!/usr/bin/env python
"""tools/record_yolo_golden.py — pin the LiteRT I/O contract for NetraYolo.kt.

Runs the .tflite on a fixed image, tries both input scalings (feed 0..255 raw
vs pre-divided /255), keeps whichever matches the .pt reference, and writes
core/fixtures/yolo/golden_v1.json with the winning graph_normalizes_input
flag, class labels from model.names, and base64 tensors for device parity.
"""
import argparse, base64, hashlib, json, pathlib
import numpy as np
from PIL import Image

ROOT = pathlib.Path(__file__).resolve().parents[1]
TFLITE = ROOT / "apps/mobile/android/app/assets/yolo26n_roi.tflite"
PT = ROOT / "core/models/netra_roi.pt"
CONF, IOU = 0.25, 0.45


def letterbox(im, size=640, pad=114):
    h, w = im.shape[:2]
    gain = min(size / w, size / h)
    # int(x + 0.5) = half-up rounding, matches Kotlin roundToInt()
    nw, nh = max(1, int(w * gain + 0.5)), max(1, int(h * gain + 0.5))
    px, py = (size - nw) // 2, (size - nh) // 2
    out = np.full((size, size, 3), pad, np.uint8)
    out[py:py + nh, px:px + nw] = np.asarray(
        Image.fromarray(im).resize((nw, nh), Image.BILINEAR))
    return out, dict(gain=gain, padX=float(px), padY=float(py))


def iou_np(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[0] + a[2], b[0] + b[2]), min(a[1] + a[3], b[1] + b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / (a[2] * a[3] + b[2] * b[3] - inter + 1e-9)


def decode(rows, meta):
    scores = rows[4:]
    if (scores < 0).any() or (scores > 1).any():
        scores = 1 / (1 + np.exp(-scores))
    cls, conf = scores.argmax(0), scores.max(0)
    keep = conf >= CONF
    boxes = np.stack([rows[0] - rows[2] / 2, rows[1] - rows[3] / 2,
                      rows[2], rows[3]], 1)[keep]
    boxes[:, 0] = (boxes[:, 0] - meta["padX"]) / meta["gain"]
    boxes[:, 1] = (boxes[:, 1] - meta["padY"]) / meta["gain"]
    boxes[:, 2] /= meta["gain"]
    boxes[:, 3] /= meta["gain"]
    dets = []
    for c in np.unique(cls[keep]):
        b, s = boxes[cls[keep] == c], conf[keep][cls[keep] == c]
        order = s.argsort()[::-1]
        kept = []
        for i in order:
            if all(iou_np(b[i], b[j]) <= IOU for j in kept):
                kept.append(i)
        dets += [dict(label=int(c), score=float(s[i]),
                      x=float(b[i, 0]), y=float(b[i, 1]),
                      w=float(b[i, 2]), h=float(b[i, 3])) for i in kept]
    return dets


def reference_pt(img_path):
    from ultralytics import YOLO
    model = YOLO(str(PT))
    r = model.predict(source=str(img_path), imgsz=640,
                      conf=CONF, iou=IOU, verbose=False)[0]
    dets = [dict(label=int(c), score=float(s), x=float(b[0]), y=float(b[1]),
                 w=float(b[2] - b[0]), h=float(b[3] - b[1]))
            for b, c, s in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf)]
    return dets, {int(k): v for k, v in model.names.items()}


def parity_ok(dets, ref):
    if abs(len(dets) - len(ref)) > 1:
        return False
    for d in ref:
        m = [x for x in dets if x["label"] == d["label"]]
        if not any(iou_np((d["x"], d["y"], d["w"], d["h"]),
                          (x["x"], x["y"], x["w"], x["h"])) >= 0.9
                   and abs(x["score"] - d["score"]) <= 0.02 for x in m):
            return False
    return True


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--img", required=True)
    img = ap.parse_args().img
    im = np.asarray(Image.open(img).convert("RGB"))
    lb, meta = letterbox(im)
    try:
        from tflite_runtime.interpreter import Interpreter
    except ImportError:
        from tensorflow.lite import Interpreter
    interp = Interpreter(model_path=str(TFLITE))
    interp.allocate_tensors()
    it, ot = interp.get_input_details()[0], interp.get_output_details()[0]
    assert it["shape"] == [1, 3, 640, 640], f"unexpected input shape {it['shape']}"
    assert ot["shape"] == [1, 9, 8400], f"unexpected output shape {ot['shape']}"

    ref, names = reference_pt(img)
    if len(ref) == 0:
        raise RuntimeError(
            "the .pt detects NOTHING on this image (conf 0.25). "
            "Use a clearer, closer photo of a package.")
    if len({d['label'] for d in ref}) < 3:
        print("WARNING: only these classes detected:",
              sorted(names[d['label']] for d in ref))
        print("         golden is still written (it pins input scaling + raw "
              "output), but decoded parity is weak —")
        print("         a photo with more visible fields would be better.")

    chosen = None
    for normalize_in_graph in (True, False):
        x = np.ascontiguousarray(lb.astype(np.float32).transpose(2, 0, 1))[None]
        if not normalize_in_graph:
            x = (x / 255.).astype(np.float32)
        interp.set_tensor(it["index"], x)
        interp.invoke()
        out = interp.get_tensor(ot["index"])[0]
        rows = out if out.shape[0] == 9 else out.T
        dets = decode(rows, meta)
        if parity_ok(dets, ref):
            chosen = (normalize_in_graph, x, rows, dets)
            break
    if chosen is None:
        raise RuntimeError("neither input scaling matches the .pt reference — "
                           "paste this output back for diagnosis")
    normalize_in_graph, x, rows, dets = chosen

    payload = dict(
        image=str(img), meta=meta,
        class_labels=[names[i] for i in range(len(names))],
        graph_normalizes_input=normalize_in_graph,
        input=dict(shape=[1, 3, 640, 640],
                   values_b64=base64.b64encode(x.tobytes()).decode(),
                   sha256=hashlib.sha256(x.tobytes()).hexdigest()),
        output=dict(shape=[1, 9, rows.shape[1]],
                    values_b64=base64.b64encode(
                        rows.astype(np.float32).tobytes()).decode(),
                    score_min=float(rows[4:].min()),
                    score_max=float(rows[4:].max())),
        decoded=dets, reference=ref, thresholds=dict(conf=CONF, iou=IOU),
    )
    out_path = ROOT / "core/fixtures/yolo/golden_v1.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(payload))
    print(f"wrote {out_path}")
    print(f"  dets={len(dets)}  ref={len(ref)}  "
          f"graph_normalizes_input={normalize_in_graph}")
    print(f"  raw score range "
          f"[{payload['output']['score_min']:.4f}, "
          f"{payload['output']['score_max']:.4f}]")
    print("  classes detected:",
          sorted(set(names[d['label']] for d in dets)))


if __name__ == "__main__":
    main()

Writing /content/NETRA/tools/record_yolo_golden.py


In [26]:
import ast
src = open('/content/NETRA/tools/record_yolo_golden.py').read()
ast.parse(src)
print('syntax OK,', len(src.splitlines()), 'lines')
assert 'transpose(2, 0, 1)' in src
print('NCHW version confirmed')

syntax OK, 161 lines
NCHW version confirmed


In [27]:
import os, shutil
from google.colab import files
from PIL import Image

os.makedirs('/content/NETRA/core/fixtures/yolo', exist_ok=True)
up = files.upload()          # pick a clear package photo (JPG or PNG)
shutil.move(list(up.keys())[0], '/content/NETRA/core/fixtures/yolo/golden_input.png')
print('saved:', Image.open('/content/NETRA/core/fixtures/yolo/golden_input.png').size)

Saving 16d4024f-5ad0-4bc2-afd5-4dbbf141d869.jpg to 16d4024f-5ad0-4bc2-afd5-4dbbf141d869.jpg
saved: (960, 1280)


In [32]:
%cd /content/NETRA
!python tools/record_yolo_golden.py --img core/fixtures/yolo/golden_input.png

/content/NETRA
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
Traceback (most recent call last):
  File "/content/NETRA/tools/record_yolo_golden.py", line 162, in <module>
    main()
    ~~~~^^
  File "/content/NETRA/tools/record_yolo_golden.py", line 130, in main
    raise RuntimeError("neither input scaling matches the .pt reference — "
                       "paste this output back for diagnosis")
RuntimeError: neither input scaling matches the .pt reference — paste this output back for diagnosis


In [33]:
# ============ DIAGNOSTIC: why .tflite parity failed ============
import numpy as np, pathlib, torch
from PIL import Image
import tensorflow as tf
from ultralytics import YOLO

ROOT = pathlib.Path('/content/NETRA')
IMG, PT, TFL = (ROOT / 'core/fixtures/yolo/golden_input.png',
                ROOT / 'core/models/netra_roi.pt',
                ROOT / 'apps/mobile/android/app/assets/yolo26n_roi.tflite')
CONF, IOU = 0.25, 0.45
NAMES = ['PACKAGE', 'PDP', 'PRICE', 'BARCODE', 'BOP']

# letterbox — identical math to the recorder
im = np.asarray(Image.open(IMG).convert('RGB'))
H, W = im.shape[:2]
gain = min(640 / W, 640 / H)
nw, nh = int(W * gain + 0.5), int(H * gain + 0.5)
padx, pady = (640 - nw) // 2, (640 - nh) // 2
lb = np.full((640, 640, 3), 114, np.uint8)
lb[pady:pady+nh, padx:padx+nw] = np.asarray(
    Image.fromarray(im).resize((nw, nh), Image.BILINEAR))

# [1] what predict() says (full ultralytics pipeline)
res = YOLO(str(PT)).predict(source=str(IMG), imgsz=640, conf=CONF,
                            iou=IOU, verbose=False)[0]
ref = [(int(c), float(s), float(b[0]), float(b[1]),
        float(b[2]-b[0]), float(b[3]-b[1]))
       for b, c, s in zip(res.boxes.xyxy, res.boxes.cls, res.boxes.conf)]
print(f'[1] .pt predict(): {len(ref)} boxes')
for li, sc, x, y, w_, h_ in ref:
    print(f'      {NAMES[li]:8s} score={sc:.3f}  xywh=({x:.0f},{y:.0f},{w_:.0f},{h_:.0f})')

# [2] raw head type + dense-vs-dense faithfulness (pins normalization too)
model = YOLO(str(PT)).model.cpu().eval()
print('\n[2] head type:', type(model.model[-1]).__name__)

pt_dense = {}
for pdiv, ptag in ((1.0, '0..255'), (255.0, '/255')):
    arr = np.ascontiguousarray((lb.astype(np.float32) / pdiv).transpose(2, 0, 1)[None])
    with torch.no_grad():
        out = model(torch.from_numpy(arr))
    if isinstance(out, (tuple, list)):
        out = out[0]
    out = out.detach().cpu().numpy()
    print(f'    torch raw head output ({ptag} input): shape={out.shape}')
    if out.ndim == 3 and out.shape[1] == 9:
        pt_dense[ptag] = out[0]                      # (9,8400)

interp = tf.lite.Interpreter(model_path=str(TFL)); interp.allocate_tensors()
it, ot = interp.get_input_details()[0], interp.get_output_details()[0]
tfl_rows = {}
print('    .tflite vs .pt raw head — max abs diff:')
for tdiv, ttag in ((1.0, 'tflite 0..255'), (255.0, 'tflite /255')):
    xin = np.ascontiguousarray((lb.astype(np.float32) / tdiv).transpose(2, 0, 1)[None])
    interp.set_tensor(it['index'], xin); interp.invoke()
    rows_t = interp.get_tensor(ot['index'])[0].copy()
    tfl_rows[ttag] = rows_t
    for ptag, o_pt in pt_dense.items():
        d = float(np.abs(rows_t - o_pt).max())
        print(f'      {ttag} vs pt {ptag}: {d:.6f}'
              + ('   <== FAITHFUL EXPORT' if d < 0.05 else ''))

# [3] decode-hypothesis table: are predict() boxes present in the raw output?
def iou_best(c, r):
    x1 = np.maximum(c[:, 0], r[0]); y1 = np.maximum(c[:, 1], r[1])
    x2 = np.minimum(c[:, 0] + c[:, 2], r[0] + r[2])
    y2 = np.minimum(c[:, 1] + c[:, 3], r[1] + r[3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    return float((inter / (c[:, 2]*c[:, 3] + r[2]*r[3] - inter + 1e-9)).max())

if ref:
    refs_lb = [(li, x*gain+padx, y*gain+pady, w_*gain, h_*gain)
               for li, _, x, y, w_, h_ in ref]
    print('\n[3] decode hypotheses (mean best-IoU vs predict() boxes):')
    for ttag, rows in tfl_rows.items():
        sc = rows[4:]
        if (sc < 0).any() or (sc > 1).any():
            sc = 1 / (1 + np.exp(-sc))
        print(f'  --- {ttag} ---')
        print(f'    box-row ranges: r0[{rows[0].min():.1f},{rows[0].max():.1f}] '
              f'r1[{rows[1].min():.1f},{rows[1].max():.1f}] '
              f'r2[{rows[2].min():.1f},{rows[2].max():.1f}] '
              f'r3[{rows[3].min():.1f},{rows[3].max():.1f}]')
        top = sc.max(0).argsort()[::-1][:5]
        for a in top:
            print(f'      top cand cls={int(sc.argmax(0)[a])} conf={sc.max(0)[a]:.3f} '
                  f'raw=({rows[0,a]:.1f},{rows[1,a]:.1f},{rows[2,a]:.1f},{rows[3,a]:.1f})')
        for hyp in ('cxcywh_px', 'xyxy_px', 'cxcywh_norm640', 'xyxy_norm640'):
            if hyp == 'cxcywh_px':
                c = np.stack([rows[0]-rows[2]/2, rows[1]-rows[3]/2, rows[2], rows[3]], 1)
            elif hyp == 'xyxy_px':
                c = np.stack([rows[0], rows[1], rows[2]-rows[0], rows[3]-rows[1]], 1)
            elif hyp == 'cxcywh_norm640':
                b = rows[:4] * 640.0
                c = np.stack([b[0]-b[2]/2, b[1]-b[3]/2, b[2], b[3]], 1)
            else:
                b = rows[:4] * 640.0
                c = np.stack([b[0], b[1], b[2]-b[0], b[3]-b[1]], 1)
            m = np.mean([iou_best(c, r) for r in refs_lb])
            print(f'    {hyp:15s}: {m:.3f}' + ('   <== MATCH' if m > 0.8 else ''))

[1] .pt predict(): 7 boxes
      PDP      score=0.921  xywh=(96,205,317,423)
      PACKAGE  score=0.874  xywh=(72,140,362,1000)
      PACKAGE  score=0.467  xywh=(486,147,451,1032)
      PDP      score=0.347  xywh=(561,317,300,183)
      PRICE    score=0.315  xywh=(547,937,316,104)
      BARCODE  score=0.294  xywh=(424,1065,103,95)
      PRICE    score=0.259  xywh=(518,1072,353,67)

[2] head type: Detect
    torch raw head output (0..255 input): shape=(1, 9, 8400)
    torch raw head output (/255 input): shape=(1, 9, 8400)
    .tflite vs .pt raw head — max abs diff:


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


      tflite 0..255 vs pt 0..255: 656.203857
      tflite 0..255 vs pt /255: 685.513184
      tflite /255 vs pt 0..255: 656.192200
      tflite /255 vs pt /255: 685.427917

[3] decode hypotheses (mean best-IoU vs predict() boxes):
  --- tflite 0..255 ---
    box-row ranges: r0[-0.2,1.0] r1[-0.0,1.0] r2[-0.0,0.9] r3[-0.0,0.6]
      top cand cls=3 conf=1.000 raw=(0.2,0.1,0.1,0.1)
      top cand cls=3 conf=1.000 raw=(0.2,0.0,0.1,0.1)
      top cand cls=3 conf=1.000 raw=(0.7,0.9,0.3,0.1)
      top cand cls=3 conf=1.000 raw=(0.6,0.9,0.2,0.0)
      top cand cls=3 conf=1.000 raw=(0.7,0.9,0.1,0.0)
    cxcywh_px      : 0.000
    xyxy_px        : 0.000
    cxcywh_norm640 : 0.421
    xyxy_norm640   : 0.191
  --- tflite /255 ---
    box-row ranges: r0[0.0,1.0] r1[0.0,1.1] r2[0.0,0.6] r3[0.0,0.8]
      top cand cls=0 conf=0.926 raw=(0.3,0.5,0.3,0.8)
      top cand cls=1 conf=0.915 raw=(0.3,0.3,0.2,0.3)
      top cand cls=1 conf=0.909 raw=(0.3,0.3,0.2,0.3)
      top cand cls=1 conf=0.884 raw=(0.3,0.

In [34]:
%%writefile /content/NETRA/tools/record_yolo_golden.py
#!/usr/bin/env python
"""tools/record_yolo_golden.py — pin the LiteRT I/O contract for NetraYolo.kt.

v2 — contract established empirically:
  input : [1,3,640,640] float32 NCHW, values ALREADY divided by 255
          (the graph does NOT normalize internally)
  output: [1,9,8400] float32 native. Rows 0-3 = cx,cy,w,h NORMALIZED [0,1]
          over the 640x640 canvas; rows 4-8 = post-sigmoid class scores.
Reference = the .pt torch model fed the EXACT same tensor (never predict(),
which letterboxes rectangularly and cannot match).
"""
import argparse, base64, hashlib, json, pathlib

import numpy as np
from PIL import Image

ROOT = pathlib.Path(__file__).resolve().parents[1]
TFLITE = ROOT / "apps/mobile/android/app/assets/yolo26n_roi.tflite"
PT = ROOT / "core/models/netra_roi.pt"
CONF, IOU, SIZE = 0.25, 0.45, 640


def letterbox(im, size=SIZE, pad=114):
    h, w = im.shape[:2]
    gain = min(size / w, size / h)
    nw, nh = max(1, int(w * gain + 0.5)), max(1, int(h * gain + 0.5))
    px, py = (size - nw) // 2, (size - nh) // 2
    out = np.full((size, size, 3), pad, np.uint8)
    out[py:py + nh, px:px + nw] = np.asarray(
        Image.fromarray(im).resize((nw, nh), Image.BILINEAR))
    return out, dict(gain=gain, padX=float(px), padY=float(py))


def iou_np(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[0] + a[2], b[0] + b[2]), min(a[1] + a[3], b[1] + b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / (a[2] * a[3] + b[2] * b[3] - inter + 1e-9)


def decode(rows_px, meta):
    """rows_px: (9, 8400); boxes in letterbox PIXELS; scores post-sigmoid."""
    scores = rows_px[4:]
    if (scores < 0).any() or (scores > 1).any():
        scores = 1 / (1 + np.exp(-scores))
    cls, conf = scores.argmax(0), scores.max(0)
    keep = conf >= CONF
    boxes = np.stack([rows_px[0] - rows_px[2] / 2,
                      rows_px[1] - rows_px[3] / 2,
                      rows_px[2], rows_px[3]], 1)[keep]
    boxes[:, 0] = (boxes[:, 0] - meta["padX"]) / meta["gain"]
    boxes[:, 1] = (boxes[:, 1] - meta["padY"]) / meta["gain"]
    boxes[:, 2] /= meta["gain"]
    boxes[:, 3] /= meta["gain"]
    dets = []
    for c in np.unique(cls[keep]):
        b = boxes[cls[keep] == c]
        s = conf[keep][cls[keep] == c]
        order = s.argsort()[::-1]
        kept = []
        for i in order:
            if all(iou_np(b[i], b[j]) <= IOU for j in kept):
                kept.append(i)
        dets += [dict(label=int(c), score=float(s[i]),
                      x=float(b[i, 0]), y=float(b[i, 1]),
                      w=float(b[i, 2]), h=float(b[i, 3])) for i in kept]
    return dets


def torch_reference(x):
    """Run the .pt on the EXACT tensor we feed the tflite."""
    import torch
    from ultralytics import YOLO
    model = YOLO(str(PT)).model.cpu().eval()
    names = {int(k): v for k, v in model.names.items()}
    with torch.no_grad():
        out = model(torch.from_numpy(x))
    if isinstance(out, (tuple, list)):
        out = out[0]
    out = out.detach().cpu().numpy()
    assert out.shape == (1, 9, 8400), f"torch raw shape {out.shape}"
    return out[0], names


def tflite_raw(x):
    try:
        from tflite_runtime.interpreter import Interpreter
    except ImportError:
        import tensorflow as tf
        Interpreter = tf.lite.Interpreter
    interp = Interpreter(model_path=str(TFLITE))
    interp.allocate_tensors()
    it = interp.get_input_details()[0]
    ot = interp.get_output_details()[0]
    assert list(it["shape"]) == [1, 3, SIZE, SIZE], f"input shape {it['shape']}"
    assert list(ot["shape"]) == [1, 9, 8400], f"output shape {ot['shape']}"
    interp.set_tensor(it["index"], x)
    interp.invoke()
    return interp.get_tensor(ot["index"])[0].copy()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--img", required=True)
    img = ap.parse_args().img
    im = np.asarray(Image.open(img).convert("RGB"))
    lb, meta = letterbox(im)
    x = np.ascontiguousarray(
        lb.astype(np.float32).transpose(2, 0, 1) / np.float32(255.0))[None]

    rows_pt, names = torch_reference(x)      # boxes in px
    rows_t = tflite_raw(x)                   # boxes normalized [0,1]
    rows_t_px = rows_t.copy()
    rows_t_px[:4] *= float(SIZE)

    fid_box = float(np.abs(rows_t_px[:4] - rows_pt[:4]).max())
    fid_scr = float(np.abs(rows_t[4:] - rows_pt[4:]).max())
    print(f"export fidelity (max|diff|): boxes {fid_box:.4f} px, "
          f"scores {fid_scr:.6f}")

    dets_pt = decode(rows_pt, meta)
    dets_t = decode(rows_t_px, meta)
    if not dets_pt:
        raise RuntimeError("torch reference decodes ZERO boxes at conf 0.25 — "
                           "try a clearer, closer photo")
    print(f"torch reference: {len(dets_pt)} boxes")
    for d in dets_pt:
        print(f"    {names[d['label']]:8s} score={d['score']:.3f} "
              f"xywh=({d['x']:.0f},{d['y']:.0f},{d['w']:.0f},{d['h']:.0f})")
    print(f"tflite decode   : {len(dets_t)} boxes")
    for d in dets_t:
        print(f"    {names[d['label']]:8s} score={d['score']:.3f} "
              f"xywh=({d['x']:.0f},{d['y']:.0f},{d['w']:.0f},{d['h']:.0f})")

    # Parity gates on conf >= 0.35 boxes only, so a borderline 0.25-gate flip
    # between float backends cannot fail an otherwise-healthy export.
    strong = [d for d in dets_pt if d["score"] >= 0.35]
    if len(dets_t) > len(dets_pt) + 3:
        raise RuntimeError("tflite produces far more boxes than torch — "
                           "paste this output back")
    for d in strong:
        m = [e for e in dets_t if e["label"] == d["label"]]
        if not any(iou_np((d["x"], d["y"], d["w"], d["h"]),
                          (e["x"], e["y"], e["w"], e["h"])) >= 0.85
                   and abs(e["score"] - d["score"]) <= 0.05 for e in m):
            raise RuntimeError(f"no tflite match for torch "
                               f"{names[d['label']]}@{d['score']:.3f} — "
                               f"paste this output back")

    if len({d["label"] for d in dets_t}) < 3:
        print("WARNING: fewer than 3 classes detected — golden is valid but "
              "parity coverage is weak; consider a photo with more fields")

    payload = dict(
        image=str(img), meta=meta,
        class_labels=[names[i] for i in range(len(names))],
        graph_normalizes_input=False,
        output_box_space="normalized_0_1",
        input=dict(shape=[1, 3, SIZE, SIZE],
                   values_b64=base64.b64encode(x.tobytes()).decode(),
                   sha256=hashlib.sha256(x.tobytes()).hexdigest()),
        output=dict(shape=[1, 9, rows_t.shape[1]],
                    values_b64=base64.b64encode(
                        rows_t.astype(np.float32).tobytes()).decode(),
                    score_min=float(rows_t[4:].min()),
                    score_max=float(rows_t[4:].max())),
        export_fidelity=dict(box_px=fid_box, score=fid_scr),
        decoded=dets_t, reference=dets_pt,
        thresholds=dict(conf=CONF, iou=IOU),
    )
    out_path = ROOT / "core/fixtures/yolo/golden_v1.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(payload))
    print(f"wrote {out_path}")
    print("  classes detected:",
          sorted(set(names[d["label"]] for d in dets_t)))


if __name__ == "__main__":
    main()

Overwriting /content/NETRA/tools/record_yolo_golden.py


In [35]:
%cd /content/NETRA
!python tools/record_yolo_golden.py --img core/fixtures/yolo/golden_input.png

/content/NETRA
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
export fidelity (max|diff|): boxes 0.0152 px, scores 0.000006
torch reference: 6 boxes
    PACKAGE  score=0.926 xywh=(72,155,361,966)
    PACKAGE  score=0.613 xywh=(449,227,499,878)
    PDP      score=0.915 xywh=(100,224,311,401)
    PDP      score=0.279 xywh=(564,318,295,183)
    PRICE    score=0.441 xywh=(554,920,305,116)
    PRICE    score=0.310 xywh=(132,789,302,304)
tflite decode   : 6 boxes
    PACKAGE  score=0.926 xywh=(72,155,361,966)
    PACKAGE  score=0.613 xywh=(449,227,499,878)
    PDP      score=0.915 xywh=(100

In [36]:
import shutil, pathlib
SRC = pathlib.Path('/content/NETRA')
DST = pathlib.Path('/content/drive/MyDrive/netra_models')
for f in ['tools/record_yolo_golden.py',
          'core/fixtures/yolo/golden_v1.json',
          'core/fixtures/yolo/golden_input.png']:
    shutil.copy(SRC / f, DST / f.split('/')[-1])
    print('copied', f)

copied tools/record_yolo_golden.py
copied core/fixtures/yolo/golden_v1.json
copied core/fixtures/yolo/golden_input.png
